# Worst Week on the Leaderboard

**Topic:** Vectorized NumPy &nbsp;·&nbsp; **Difficulty:** Medium

A training-run dashboard tracks validation accuracy for 200 experiments over 500
steps each. The team wants to rank runs by how badly they *regressed*: for each
run, the largest drop from any previous peak to a later point.

Written as a loop this is 100,000 comparisons in total, one per step of every
run. Written properly it is one call to a NumPy function most people forget
exists.

## The data (created in the next cell)

- `curves` `(200, 500)`: validation accuracy per run per step. Not monotone.
- Every run starts near 0.5 and generally improves, but some collapse.

## Your task

With **NumPy only** and **no Python loop over steps**:

1. For each run, compute the running maximum of the curve up to and including
   each step.
2. The drawdown at a step is `running_max - value` at that step.
3. Set `max_drawdowns` to a length-200 array holding each run's largest drawdown.

Hint: `np.maximum.accumulate` does step 1 along an axis in one call.

## Evaluation

Your `max_drawdowns` is checked in two scalars, its sum and a fixed random projection, against reference values to a tolerance of 1e-9.


In [ ]:
import numpy as np

rng = np.random.default_rng(23)
n_runs, n_steps = 200, 500
trend = np.linspace(0.5, 0.9, n_steps)[None, :] * rng.uniform(0.8, 1.1, (n_runs, 1))
noise = rng.normal(0, 0.012, (n_runs, n_steps)).cumsum(axis=1) * 0.15
curves = trend + noise
collapse = rng.random(n_runs) < 0.25
where = rng.integers(150, 450, n_runs)
for i in np.flatnonzero(collapse):
    curves[i, where[i]:] -= rng.uniform(0.15, 0.45)
print(f"curves {curves.shape}, {int(collapse.sum())} runs collapse")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "max_drawdowns" not in dir():
    raise NameError("Define `max_drawdowns` in the solution cell before running this cell.")
answers = {"max_drawdowns": max_drawdowns}

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("max_drawdowns", [200], 15.815980066136774, 1.0633034493123397, 1e-09),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

print("=" * 56)
if _ok:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

`np.maximum.accumulate(curves, axis=1)` gives the running peak for every run at
once. Subtract, then take the row-wise max. No loop over steps, and the whole
thing is three passes over the array.

```python
import numpy as np

running_peak = np.maximum.accumulate(curves, axis=1)
max_drawdowns = (running_peak - curves).max(axis=1)
```